# **Machine Learning: aplicado a dados imobilísticos do Paquistão**
Equipe: Gabriel Souza Dunkel, Guilherme Henriques Almeida, Luca Torres Villela

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
import numpy as np

pd.options.display.max_rows = None
df = pd.read_csv('pakistan_house_prices.csv')
df.info()

### **Tipo de cada coluna:**
- unnamed: Categórica e Nominal (um tipo de id)
- property_type: Categórica e Nominal
- price: Numérica e Razão
- location: Categórica e Nominal
- city: Categórica e Nominal
- baths e bedrooms: Numérica e Razão
- purpose: Categórica e Nominal (For sale, for rent)
- Area_in_marla: Numérica e Razão

### **Tratamentos iniciais necessários**
- A função *info()* nos mostrou que não há dados nulos ou faltantes. Por isso não removeremos linhas e não faremos inputação
- Removeremos a coluna "unnamed" pois é um tipo de ID e não vai ajudar na análise ou predicão de dados
- Percebemos então que existem alguns dados onde o valor da area e da quantidade de quartos é 0
- Decidimos remover esses dados, não achamos que valia a pena fazer a análise para "prever" os valores
- 10 linhas onde a "Area_in_Marla" = 0, e 294 linhas onde "bedrooms" = 0
- Finalmente faremos a conversão de Marla para metros quadrados na coluna "Area_in_Marla" para uma melhor compreensão
- Valores pareciam meio absurdos, com isso assumimos que eles estavam em Rupees (moeda local do Paquistão). Com isso, fizemos a conversão para melhor compreensão dos dados



In [ ]:
clean_df = df.copy()
clean_df.drop(columns=['Unnamed: 0'], inplace=True)
clean_df = clean_df[clean_df['Area_in_Marla'] != 0] #remover linhas com area 0  (10 linhas)
clean_df = clean_df[clean_df['bedrooms'] != 0] #remover linhas com quartos 0    (294 linhas)
clean_df['area'] = df['Area_in_Marla'] * 25.292852 #converter de Marla pra metro quadrado
clean_df['area'] = clean_df['area'].round(0).astype(int) #converter valor pra inteiro pq normalmente é assim em anuncios de imóveis 
clean_df['price'] = df['price'] / 52 # converter de Rupee para Reais
clean_df['price'] = clean_df['price'].round(2) # arredondamento para 2 casas decimais
clean_df.drop(columns=['Area_in_Marla'], inplace=True)

print("Número de linhas removidas na limpeza dos dados:")
print(len(df) - len(clean_df))

print("Número de linhas restantes após limpeza dos dados:")
print(len(clean_df))

clean_df.head(10)

### **Observação de Outliers**
Fazendo umas queries e observando a média de metros quadrados por tipo de propriedade, observamos alguns valores absurdos:

#### Variação do Tamanho (area) por Tipo e Localização:


In [ ]:
%matplotlib inline
sns.set_theme(style="whitegrid")

# A melhor forma de ver "como varia" é com agregações e gráficos.

# 1: Média de área por TIPO de propriedade
print("  Média de área por tipo de propriedade:")
media_area_por_tipo = clean_df.groupby('property_type')['area'].mean().sort_values(ascending=False)
print(media_area_por_tipo)
print("\n")

# 2: Média de área por CIDADE
# (Usar 'city' é geralmente melhor que 'location', que pode ter valores demais)
print("  Média de área por cidade:")
media_area_por_cidade = clean_df.groupby('city')['area'].mean().sort_values(ascending=False)
print(media_area_por_cidade.all)
print("\n")


In [ ]:
#Breve query que mostra as 'n' maiores áreas por tipo de propriedade:
area_maxima = clean_df[clean_df['property_type'] == 'Flat']['area'].nlargest(10)
area_maxima

### Decidimos então fazer um box plot para verificar os outliers

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='property_type', y='area', data=clean_df)
plt.yscale('log')
plt.show()

### **Tratamento efetivo dos outliers**

In [ ]:
#função que remove os outliers
def remove_outliers_by_group(df, column, group, lower_q=0.1, upper_q=0.9):
    return df.groupby(group, group_keys=False).apply(
        lambda x: x[(x[column] > x[column].quantile(lower_q)) &
                    (x[column] < x[column].quantile(upper_q))]
    )

df_outliers = remove_outliers_by_group(clean_df, 'area', 'property_type')

In [ ]:
#criação da coluna preço por sqm para reduzir enviezamento de resultados
df_outliers['price_per_sqm'] = df_outliers['price'] / df_outliers['area']
df_outliers['price_per_sqm'] = df_outliers['price_per_sqm'].round(2)

### Novo Boxplot depois de remover os outliers

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='property_type', y='area', data=df_outliers)
plt.yscale('log')
plt.show()

Mesmo com a remoção de outliers, House e Rooms ainda apresentam caudas. Consideramos que House faz sentido por causa de eventuais mansões, mas que em rooms deve haver algo errado, então fomos investigar.

In [ ]:
#Breve query que mostra as 'n' maiores áreas por tipo de propriedade:
pd.set_option('display.max_rows', None)
topn = df_outliers[df_outliers['property_type'] == 'Room'].sort_values(by='price_per_sqm', ascending=False).head(12)
topn

Observando essa query vemos que valores absurdos pra quartos surgem a partir dos $32 por m2. Com isso, vamos removê-los, são apenas 10 linhas que vão mais atrapalhar do que ajudar.

In [ ]:
final_df = df_outliers[~((df_outliers['property_type'] == 'Room') & (df_outliers['price_per_sqm'] > 40))]
final_df[final_df['property_type'] == 'Room']['price_per_sqm'].max()

### Mesmo depois de todos esses tratamentos, ainda há um problema com os property_type 'Room'
- quando calculamos a média das áreas, para todos outros tipos de propriedades as metragens fazem sentido menos para os quartos.

In [ ]:
print("  Média de área por tipo de propriedade:")
media_area_por_tipo = final_df.groupby('property_type')['area'].mean().sort_values(ascending=False)
print(media_area_por_tipo)
print("\n")

- Por grande parte dos dados 'Room' se encaixarem no grupo 'Área de valor absurdo' e por sí só rooms serem poucos valores (140 linhas) tinhamos apenas 2 opções plausíveis: Dropar todos os rooms ou sintetizar o valor da área.
- A título de aprendizagem, decidimos sintetizar o valor da área usando os valores maximos e minimos e garantindo que o tamanho dos rooms estejam num intervalo plausível.

In [ ]:
mean_ppsqm_flat = final_df.loc[final_df['property_type'] == 'Flat', 'price_per_sqm'].mean()

before = final_df.loc[final_df['property_type'] == 'Room', 'area'].copy()

mask = (final_df['property_type'] == 'Room') & ((final_df['area'] < 5) | (final_df['area'] > 50))

# Normalizar os preços dos Rooms inválidos (escala 0–1)
min_price = final_df.loc[mask, 'price'].min()
max_price = final_df.loc[mask, 'price'].max()

# Calcular nova área proporcional ao preço
#    - Preços baixos -> área perto de 5 m²
#    - Preços altos -> área perto de 50 m²
final_df.loc[mask, 'area'] = 5 + (
    (final_df.loc[mask, 'price'] - min_price) /
    (max_price - min_price)
) * (50 - 5)

final_df['price_per_sqm'] = final_df['price'] / final_df['area']

before_count = before.shape[0]
after_count = final_df.loc[final_df['property_type'] == 'Room'].shape[0]

before_mean = before.mean()
after_mean = final_df.loc[final_df['property_type'] == 'Room', 'area'].mean()

print(f'🧾 Rooms antes: {before_count} | depois: {after_count}')
print(f'📐 Média de área antes: {before_mean:.2f} | depois: {after_mean:.2f}')

final_df['area'] = final_df['area'].round(0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(before, bins=30, kde=True, ax=axes[0], color='tomato')
axes[0].set_title('Distribuição de Áreas - "Room" (Antes)')
axes[0].set_xlabel('Área (m²)')

sns.histplot(final_df.loc[final_df['property_type'] == 'Room', 'area'],
             bins=30, kde=True, ax=axes[1], color='seagreen')
axes[1].set_title('Distribuição de Áreas - "Room" (Depois)')
axes[1].set_xlabel('Área (m²)')

plt.tight_layout()
plt.show()

### **Histograma agora muito melhor distribuido**
- Nova média é de 17.6 para quartos

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='property_type', y='area', data=final_df)
plt.yscale('log')
plt.show()

### Verificação de quantos quartos estão For Sale

In [ ]:
rooms_for_sale = final_df[
    (final_df['property_type'] == 'Room') & 
    (final_df['purpose'] == 'For Sale')
]
rooms_for_sale


In [ ]:
final_df.loc[78191, 'purpose'] = 'For Rent'
rooms_for_sale = final_df[
    (final_df['property_type'] == 'Room') & 
    (final_df['purpose'] == 'For Sale')
]
rooms_for_sale
len(rooms_for_sale)

Apenas 1, consideramos que foi um erro e o convertemos em 'For Rent'

### Análise Exploratória dos Dados Geral

In [ ]:
pd.reset_option('display.max_rows')
clean_df = final_df.copy()
clean_df.info()
clean_df.describe()

In [ ]:
%matplotlib inline
media_area_por_tipo = (
    clean_df.groupby('property_type')['area']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
plt.figure(figsize=(8, 5))
sns.barplot(data=media_area_por_tipo, x='area', y='property_type')
plt.title('Média de Área por Tipo de Propriedade')
plt.xlabel('Área média (m²)')
plt.ylabel('Tipo de Propriedade')
plt.tight_layout()
plt.show()


media_area_por_cidade = (
    clean_df.groupby('city')['area']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
plt.figure(figsize=(8, 5))
sns.barplot(data=media_area_por_cidade, x='area', y='city')
plt.title('Média de Área por Cidade')
plt.xlabel('Área média (m²)')
plt.ylabel('Cidade')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(20, 6))
sns.histplot(data=clean_df, x='price', bins=100, kde=True, hue='purpose', element='step')
plt.title('Distribuição de Preços')
# plt.xlim(0, 10000)
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(data=clean_df, x='area', bins=50, kde=True, hue='purpose', element='step')
plt.title('Distribuição de Áreas')
plt.xlim(0, 600)
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(data=clean_df, x='price_per_sqm', bins=50, kde=True, hue='purpose', element='step')
plt.title('Distribuição de Preço por m²')
plt.show()

### Separação de 'For Rent' e 'For Sale'

In [ ]:
rent_df = clean_df[clean_df['purpose'] == 'For Rent']
sale_df = clean_df[clean_df['purpose'] == 'For Sale']

plt.figure(figsize=(20, 6))
sns.histplot(rent_df['price'], bins=1000, kde=True)
plt.title('Distribuição de Preços (For Rent)')
plt.xlim(0, 10000)
media = rent_df['price'].mean()
plt.axvline(media, color='red', linestyle='--', linewidth=2, label=f'Média: {media:.2f}')
plt.legend()
plt.show()
plt.figure(figsize=(20, 6))
sns.histplot(sale_df['price'], bins=100, kde=True)
plt.title('Distribuição de Preços (For Sale)')
media = sale_df['price'].mean()
plt.axvline(media, color='red', linestyle='--', linewidth=2, label=f'Média: {media:.2f}')
plt.legend()
plt.show()

#### 1. Preço Médio dos Imóveis:

In [ ]:
# Calcula o preço médio por tipo para cada propósito
preco_medio_sale = sale_df.groupby('property_type')['price'].mean()
preco_medio_rent = rent_df.groupby('property_type')['price'].mean()

# Junta os dois DataFrames em um único
comparativo = pd.DataFrame({
    'For Sale': preco_medio_sale,
    'For Rent': preco_medio_rent
}).dropna()  # remove tipos que só existem em um dos dois

# Ordena pelo preço médio de venda
comparativo = comparativo.sort_values('For Sale', ascending=False)

# Plot
plt.figure(figsize=(14, 7))
ax = comparativo.plot(
    kind='bar',
    figsize=(14, 7),
    color=['steelblue', 'orange'],
    edgecolor='black'
)

plt.title('Preço Médio por Tipo de Propriedade (For Sale vs For Rent)', fontsize=16)
plt.xlabel('Tipo de Propriedade', fontsize=12)
plt.ylabel('Preço Médio', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Purpose')

# Formatar eixo Y como moeda
formatter = mticker.FuncFormatter(lambda x, p: f'${x:,.0f}')
ax.yaxis.set_major_formatter(formatter)

# Adicionar rótulos nas barras
for container in ax.containers:
    ax.bar_label(container, fmt='${:,.0f}', label_type='edge', fontsize=9, padding=2)

plt.tight_layout()
plt.show()
print('Preço médio de aluguel de quartos')
print(rent_df[rent_df['property_type'] == 'Room']['price'].mean().round(2))


#### 2. Distribuição dos Tipos de Propriedades:

In [ ]:
distribuicao_por_tipo = (
    clean_df.groupby(['property_type', 'purpose'])
    .size()
    .unstack(fill_value=0)
    .sort_values(by='For Sale', ascending=False)
)

# Plotar barras lado a lado
plt.figure(figsize=(14, 7))
ax = distribuicao_por_tipo.plot(
    kind='bar',
    figsize=(14, 7),
    color=['steelblue', 'orange'],
    edgecolor='black'
)

plt.title('Distribuição por Tipo de Propriedade (For Sale vs For Rent)', fontsize=16)
plt.xlabel('Tipo de Propriedade', fontsize=12)
plt.ylabel('Quantidade', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Purpose')

# Formatar eixo Y com separador de milhar
formatter = mticker.FuncFormatter(lambda x, p: f'{x:,.0f}')
ax.yaxis.set_major_formatter(formatter)

# Adicionar rótulos de valor nas barras
for container in ax.containers:
    ax.bar_label(container, fmt='{:.0f}', label_type='edge', fontsize=9, padding=2)

plt.tight_layout()
plt.show()

### Distribuição de tipo de propriedade por Cidade

In [ ]:
contagem_por_cidade_tipo = (
    clean_df.groupby(['city', 'property_type'])
    .size()
    .unstack(fill_value=0)
)

# Exibir a tabela resumida (opcional)
print(contagem_por_cidade_tipo)

# Plotar o gráfico empilhado
contagem_por_cidade_tipo.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 7),
    colormap='tab20' 
)

plt.title('Distribuição de Tipos de Propriedades por Cidade', fontsize=16)
plt.xlabel('Cidade', fontsize=12)
plt.ylabel('Número de Propriedades', fontsize=12)
#plt.yscale('log')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Tipo de Propriedade', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


#### 3. Média de preços por Cidade:

In [ ]:
# Calcular média de preço por cidade e propósito
media_preco_cidade_purpose = (
    clean_df.groupby(['city', 'purpose'])['price']
    .mean()
    .unstack(fill_value=0)
    .sort_values(by='For Sale', ascending=False)
)

print("Média de preço de propriedades por cidade (For Sale vs For Rent):")
print(media_preco_cidade_purpose)
print("\n")

# Plotar gráfico comparativo
plt.figure(figsize=(14, 7))
ax = media_preco_cidade_purpose.plot(
    kind='bar',
    figsize=(14, 7),
    color=['royalblue', 'orange'],
    edgecolor='black'
)

plt.title('Média de Preço de Propriedades por Cidade (For Sale vs For Rent)', fontsize=16)
plt.xlabel('Cidade', fontsize=12)
plt.ylabel('Preço Médio', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Purpose')

# Formatar eixo Y como moeda
formatter = mticker.FuncFormatter(lambda x, p: f'{x:,.2f}')
ax.yaxis.set_major_formatter(formatter)

# Adicionar rótulos de valores nas barras
for container in ax.containers:
    ax.bar_label(container, fmt='{:.2f}', label_type='edge', fontsize=9, padding=2)

plt.tight_layout()
plt.show()


#### Características das Propriedades (simples):

#### 4. Média de Quartos e Banheiros:

In [ ]:
media_comodos = clean_df[['bedrooms', 'baths']].mean().round(2)
# mediana_comodos = clean_df[['bedrooms', 'baths']].median()

print(media_comodos)
print("\n")
# print(mediana_comodos)
# print("\n")

### Scatter plot de preço por área, dividido por tipo de propriedade

In [ ]:
# Criar os dois gráficos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)

# Scatter For Sale
sns.scatterplot(
    data=sale_df,
    x='area', y='price', hue='property_type',
    ax=axes[0], alpha=0.7, edgecolor=None
)
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_title('For Sale: Área vs Preço')

# Scatter For Rent
sns.scatterplot(
    data=rent_df,
    x='area', y='price', hue='property_type',
    ax=axes[1], alpha=0.7, edgecolor=None
)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title('For Rent: Área vs Preço')

# Legenda e rótulos
for ax in axes:
    ax.set_xlabel('Área (log)')
    ax.set_ylabel('Preço (log)')
    ax.grid(True, which='both', linestyle='--', alpha=0.3)

# Ajusta layout e legenda fora do gráfico
plt.tight_layout()
plt.legend(title='Property Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### Correlação entre variáveis For Sale

In [ ]:
plt.figure(figsize=(8, 6))
correlation = sale_df[['price', 'area', 'bedrooms', 'baths']].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm')
plt.title('Correlação entre Variáveis (For Sale)')
plt.show()

### Top 10 - Bairros mais caros

In [ ]:
top_locs = clean_df.groupby('location')['price_per_sqm'].mean().sort_values(ascending=False).head(10)
sns.barplot(x=top_locs.values, y=top_locs.index)
plt.title('Top 10 Localizações por Preço Médio/m²')

Os tipos de propriedades dominantes nessas cidades são Flats e Houses

### **Salvando dataset como CSV para fazer as predições em outros arquivos**

#### Salvar dados de aluguel 

In [ ]:
#rent_df.to_csv('dataset_tratado.csv', index=False)

#### Salvar dados de venda

In [ ]:
sale_df.to_csv('dataset_tratado.csv', index=False)